In [1]:
# --- CÀI ĐẶT THƯ VIỆN CẦN THIẾT ---
import sys
# Cài pydub (xử lý âm thanh) và scipy
!{sys.executable} -m pip install pydub scipy plotly

import os
import numpy as np
from scipy.io import wavfile
from pydub import AudioSegment

# Cấu hình đường dẫn
current_dir = os.getcwd()
# Thư mục chứa dữ liệu gốc (120GB)
RAW_AUDIO_DIR = os.path.abspath(os.path.join(current_dir, '..', '..', 'data', 'raw_data'))
# Thư mục sẽ lưu file đã xử lý (Tạo mới nếu chưa có)
PROCESSED_DIR = os.path.abspath(os.path.join(current_dir, 'Audio_Processed'))

if not os.path.exists(PROCESSED_DIR):
    os.makedirs(PROCESSED_DIR)
    print(f"✅ Đã tạo thư mục lưu kết quả: {PROCESSED_DIR}")
else:
    print(f"📂 Thư mục lưu kết quả: {PROCESSED_DIR}")

     --------------------------------------- 15.6/15.6 MB 28.4 MB/s eta 0:00:00
✅ Đã tạo thư mục lưu kết quả: d:\TN-AI\automatic-depression-detector-main\automatic-depression-detector-main\preprocessing\Audio\Audio_Processed


c:\Users\LENOVO\miniconda3\envs\depression_env\lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


In [2]:
# --- HÀM CẮT BỎ KHOẢNG LẶNG (TỐI ƯU HÓA) ---
def remove_silence_and_save(participant_id):
    # 1. Tìm file âm thanh gốc
    input_path = os.path.join(RAW_AUDIO_DIR, f"{participant_id}_P", f"{participant_id}_AUDIO.wav")
    output_path = os.path.join(PROCESSED_DIR, f"{participant_id}_AUDIO_p.wav")
    
    if not os.path.exists(input_path):
        return False, "File not found"
    
    if os.path.exists(output_path):
        return True, "Already processed" # Bỏ qua nếu đã làm rồi

    try:
        # 2. Đọc file
        fs, signal = wavfile.read(input_path)
        
        # 3. Chuẩn hóa (Normalize) về khoảng [-1, 1]
        # Lưu ý: wavfile đọc ra int16 hoặc float32
        if signal.dtype == np.int16:
            signal = signal / (2**15)
        elif signal.dtype == np.int32:
            signal = signal / (2**31)
            
        # 4. Cắt đoạn nhỏ (1 giây)
        signal_len = len(signal)
        segment_size = fs * 1 # 1 giây
        
        # Tính năng lượng của từng đoạn
        segments = [signal[x:x+segment_size] for x in range(0, signal_len, segment_size)]
        energies = [(s**2).sum() / len(s) for s in segments if len(s) > 0]
        
        # 5. Ngưỡng lọc (50% của năng lượng trung bình)
        if len(energies) == 0: return False, "Empty audio"
        thres = 0.5 * np.median(energies)
        
        # 6. Giữ lại các đoạn có tiếng nói
        keep_segments = [s for i, s in enumerate(segments) if i < len(energies) and energies[i] > thres]
        
        if len(keep_segments) == 0: return False, "All silence"
        
        # 7. Nối lại thành file mới
        new_signal = np.concatenate(keep_segments)
        
        # 8. Lưu file (Chuyển lại về int16 để nghe được trên mọi máy)
        wavfile.write(output_path, fs, (new_signal * 32767).astype(np.int16))
        
        return True, "Success"
        
    except Exception as e:
        return False, str(e)

# --- CHẠY THỬ NGHIỆM TRÊN 1 FILE (ID 300) ---
print("🧪 Đang chạy thử nghiệm trên bệnh nhân 300...")
success, msg = remove_silence_and_save(300)
print(f"Kết quả ID 300: {msg}")

🧪 Đang chạy thử nghiệm trên bệnh nhân 300...
Kết quả ID 300: Success


c:\Users\LENOVO\miniconda3\envs\depression_env\lib\site-packages\ipykernel_launcher.py:15: WavFileWarning: Reached EOF prematurely; finished at 20752044 bytes, expected 166016044 bytes from header.
  from ipykernel import kernelapp as app


In [3]:
# --- VÒNG LẶP XỬ LÝ TOÀN BỘ DATASET ---
import time

# Danh sách ID bệnh nhân (từ 300 đến 492)
# Lưu ý: Một số ID có thể bị thiếu trong dataset gốc, code sẽ tự bỏ qua
start_id = 300
end_id = 492

print(f"🚀 BẮT ĐẦU XỬ LÝ TỪ ID {start_id} ĐẾN {end_id}...")
start_time = time.time()
count_success = 0
count_error = 0

for p_id in range(start_id, end_id + 1):
    # Gọi hàm xử lý đã định nghĩa ở trên
    success, msg = remove_silence_and_save(p_id)
    
    if success:
        if msg == "Success":
            print(f"✅ ID {p_id}: Đã xử lý xong.")
            count_success += 1
        else:
            # Đã làm rồi thì không in ra để đỡ rối mắt
            pass 
    else:
        if msg != "File not found": # Chỉ báo lỗi nếu file có tồn tại mà không xử lý được
            print(f"⚠️ ID {p_id}: Lỗi - {msg}")
            count_error += 1

end_time = time.time()
duration = (end_time - start_time) / 60

print("-" * 30)
print(f"🎉 HOÀN TẤT! Tổng thời gian: {duration:.2f} phút.")
print(f"📊 Thành công: {count_success} file.")
print(f"❌ Lỗi: {count_error} file.")
print(f"📂 Kiểm tra folder: {PROCESSED_DIR}")

🚀 BẮT ĐẦU XỬ LÝ TỪ ID 300 ĐẾN 492...


c:\Users\LENOVO\miniconda3\envs\depression_env\lib\site-packages\ipykernel_launcher.py:15: WavFileWarning: Reached EOF prematurely; finished at 26364844 bytes, expected 210918444 bytes from header.
  from ipykernel import kernelapp as app


✅ ID 301: Đã xử lý xong.
✅ ID 302: Đã xử lý xong.


c:\Users\LENOVO\miniconda3\envs\depression_env\lib\site-packages\ipykernel_launcher.py:15: WavFileWarning: Reached EOF prematurely; finished at 24281644 bytes, expected 194252844 bytes from header.
  from ipykernel import kernelapp as app
c:\Users\LENOVO\miniconda3\envs\depression_env\lib\site-packages\ipykernel_launcher.py:15: WavFileWarning: Reached EOF prematurely; finished at 31529644 bytes, expected 252236844 bytes from header.
  from ipykernel import kernelapp as app


✅ ID 303: Đã xử lý xong.
✅ ID 304: Đã xử lý xong.


...

✅ ID 489: Đã xử lý xong.
✅ ID 490: Đã xử lý xong.


c:\Users\LENOVO\miniconda3\envs\depression_env\lib\site-packages\ipykernel_launcher.py:15: WavFileWarning: Reached EOF prematurely; finished at 28214444 bytes, expected 225715244 bytes from header.
  from ipykernel import kernelapp as app


✅ ID 491: Đã xử lý xong.


c:\Users\LENOVO\miniconda3\envs\depression_env\lib\site-packages\ipykernel_launcher.py:15: WavFileWarning: Reached EOF prematurely; finished at 29200044 bytes, expected 233600044 bytes from header.
  from ipykernel import kernelapp as app


✅ ID 492: Đã xử lý xong.
------------------------------
🎉 HOÀN TẤT! Tổng thời gian: 0.72 phút.
📊 Thành công: 187 file.
❌ Lỗi: 0 file.
📂 Kiểm tra folder: d:\TN-AI\automatic-depression-detector-main\automatic-depression-detector-main\preprocessing\Audio\Audio_Processed
